# 03 - 在 Colab A100 上训练 GFlowNet
使用 Transformer 策略和 Trajectory Balance 目标进行 PyTorch 混合精度训练，保存并重载检查点，最后生成 Alpha 因子池。完整说明见 `docs/运行手册.md`。

In [ ]:
from pathlib import Path
import os
if not Path('configs/training_config.yaml').exists():
    repo=os.environ.get('ALPHAMINING_REPO_URL','https://github.com/JacksonChiy/AlphaMining-GFlowNet-AlphaEval.git')
    !git clone $repo
    %cd AlphaMining-GFlowNet-AlphaEval

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import torch
print('PyTorch 版本:',torch.__version__)
print('CUDA 状态:',torch.cuda.is_available(),'CUDA 运行时:',torch.version.cuda)
assert torch.cuda.is_available(),'请启用 Colab A100 GPU 运行时'
p=torch.cuda.get_device_properties(0)
print('GPU 型号:',p.name)
print('GPU 显存（GB）:',round(p.total_memory/1024**3,2))
assert 'A100' in p.name.upper(),f'正式训练要求 A100；当前检测到 {p.name}'

In [ ]:
import yaml
config=yaml.safe_load(open('configs/training_config.yaml',encoding='utf-8'))
config

In [ ]:
from src.gflownet.run_training import run
run('configs/training_config.yaml', require_a100=True, pool_size=100)

## 检查点重载测试

In [ ]:
import pandas as pd
from src.gflownet import GFlowNetTrainer, RewardEvaluator
daily=pd.read_pickle(config['dataset']['output'])
evaluator=RewardEvaluator(daily,**config['reward'])
loaded=GFlowNetTrainer.load_checkpoint('checkpoints/gflownet_best.pt',evaluator,device='cuda')
expression,_,tokens=loaded.sample_trajectory(greedy=True)
print('重载成功:',expression)
print('Token 序列:',tokens)
assert Path('results/alpha_pool.csv').exists()